# BioBERT Triage — SMOTE + Augmentation (Imbalance Mitigation)

Addresses **9:1 class skew** (Level 3 vs Level 1) via:
1. Stratified train/val split
2. Text augmentation on Levels 1–2
3. Oversampling to `TARGET_COUNTS`
4. WeightedRandomSampler during training
5. Per-class F1 / recall metrics

**Run only in Google Colab with GPU.** See `../COLAB_RUN_GUIDE.md` in the repo.

In [ ]:
# Cell 1 — Install dependencies (runs before Drive mount)
!pip install -q transformers datasets accelerate torch pandas scikit-learn imbalanced-learn nlpaug matplotlib seaborn evaluate huggingface_hub

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2 — Mount Drive and set paths
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

# Option A: clone repo / upload imbalance-mitigation folder to Drive
HELPERS_DIR = "/content/drive/MyDrive/imbalance-mitigation/colab"  # <-- EDIT
sys.path.insert(0, HELPERS_DIR)

# Option B: if running from cloned GitHub repo in /content
# sys.path.insert(0, "/content/FinalYearProject/fine-tuned-biobert/imbalance-mitigation/colab")

try:
    import paths
    TRAIN_CSV = paths.TRAIN_CSV
    COMPLAINTS_CSV = paths.COMPLAINTS_CSV
    OUTPUT_DIR = paths.OUTPUT_DIR
except ImportError:
    DATA_DIR = "/content/drive/MyDrive/triagegeist/data"  # <-- EDIT
    TRAIN_CSV = f"{DATA_DIR}/train.csv"
    COMPLAINTS_CSV = f"{DATA_DIR}/chief_complaints.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/fine_tuned_biobert_triage_smote"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Train CSV: {TRAIN_CSV}")
print(f"Complaints CSV: {COMPLAINTS_CSV}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3 — Merge data and inspect class distribution
import matplotlib.pyplot as plt
import pandas as pd

import config
from data_merge import merge_and_encode, stratified_split, print_distribution

df = merge_and_encode(TRAIN_CSV, COMPLAINTS_CSV)
print(f"Label mapping (code -> acuity): {config.LABEL_TO_ACUITY}")
print_distribution(df, "Full dataset (before split)")

train_df, val_df = stratified_split(df)
print_distribution(train_df, "Train (before augmentation)")
print_distribution(val_df, "Validation (held out)")

In [ ]:
# Cell 4 — Augment minority classes + oversample
from augmentation import augment_minority_classes
from imbalance import balance_dataset
from data_merge import print_distribution

print("Augmenting Levels 1 and 2...")
train_aug = augment_minority_classes(train_df)
print(f"Train rows after augmentation: {len(train_aug):,} (was {len(train_df):,})")

print(f"\nOversampling to TARGET_COUNTS: {config.TARGET_COUNTS}")
train_balanced = balance_dataset(train_aug, config.TARGET_COUNTS)
print_distribution(train_balanced, "Train (after augment + balance)")

In [ ]:
# Cell 5 — Visualise before/after class counts
def bar_counts(frame, title):
    counts = frame["acuity_level"].value_counts().sort_index()
    plt.figure(figsize=(8, 4))
    plt.bar(counts.index.astype(str), counts.values, color="#166534")
    plt.xlabel("Acuity level")
    plt.ylabel("Count")
    plt.title(title)
    plt.show()

bar_counts(train_df, "Train BEFORE augmentation/balance")
bar_counts(train_balanced, "Train AFTER augmentation/balance")

In [ ]:
# Cell 6 (OPTIONAL) — Embedding-space SMOTE analysis
# Set config.USE_EMBEDDING_SMOTE = True to run; default pipeline skips this.

if config.USE_EMBEDDING_SMOTE:
    import numpy as np
    import torch
    from transformers import AutoModel, AutoTokenizer
    from imbalance import embedding_smote_oversample

    print("Encoding texts with frozen BioBERT for SMOTE...")
    enc_model = AutoModel.from_pretrained(config.MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")
    enc_tok = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    enc_model.eval()

    texts = train_aug["text"].tolist()
    embs = []
    batch_size = 32
    device = next(enc_model.parameters()).device
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = enc_tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = enc_model(**inputs)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    embeddings = np.vstack(embs)

    train_balanced = embedding_smote_oversample(
        train_aug, embeddings, config.TARGET_COUNTS, k_neighbors=config.SMOTE_K_NEIGHBORS
    )
    print_distribution(train_balanced, "Train (after embedding SMOTE)")
else:
    print("Skipping embedding SMOTE (USE_EMBEDDING_SMOTE=False). Using augment + random oversample.")

In [ ]:
# Cell 7 — Tokenize for Hugging Face
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

train_ds = Dataset.from_pandas(train_balanced[["text", "label"]])
val_ds = Dataset.from_pandas(val_df[["text", "label"]])

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=config.MAX_LENGTH,
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)
train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)

print(f"Train tokens: {len(train_tok):,}  |  Val tokens: {len(val_tok):,}")

In [ ]:
# Cell 8 — Fine-tune with WeightedRandomSampler + per-class metrics
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from imbalance import compute_class_weights
from metrics import compute_metrics

num_labels = len(config.LABEL_TO_ACUITY)
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)

labels_array = np.array(train_tok["labels"])
sample_weights = compute_class_weights(pd.DataFrame({"label": labels_array}))
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, train_sampler=None, **kwargs):
        self._train_sampler = train_sampler
        super().__init__(*args, **kwargs)

    def get_train_dataloader(self):
        if self._train_sampler is None:
            return super().get_train_dataloader()
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self._train_sampler,
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=config.RANDOM_SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    train_sampler=sampler,
)

print("Starting training...")
trainer.train()

In [ ]:
# Cell 9 — Final evaluation + confusion matrix
from metrics import plot_confusion_matrix, print_classification_report

pred = trainer.predict(val_tok)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print_classification_report(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred)
plt.show()

print("\nKey clinical metrics:")
from sklearn.metrics import recall_score
for i, name in [(0, "Level 1 Red"), (1, "Level 2 Orange")]:
    r = recall_score(y_true == i, y_pred == i, zero_division=0)
    print(f"  Recall {name}: {r:.3f}")

In [ ]:
# Cell 10 — Save model to Google Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")
print("Download model.safetensors from Drive to use in Curatio (set MODEL_PATH).")